In [2]:
!pip install -qU chromadb sentence-transformers

In [2]:
#Import and Create a Client
import chromadb

# This creates a database that will be saved on your hard drive
client = chromadb.PersistentClient(path="./my_chroma_db")

In [3]:
# Undrestanding what an embedding Function Does
from chromadb.utils import embedding_functions

# This is the "brain" that converts text to numbers
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

In [4]:
# Create a Collection
# A collection is like a "table" in traditional databases
collection = client.get_or_create_collection(
    name="my_recipes1",  # Name of your collection
    embedding_function=sentence_transformer_ef,  # How to convert text
    metadata={"description": "My favorite recipes"}  # Info about the collection itself
)

print(f"Collection name: {collection.name}")
print(f"Collection metadata: {collection.metadata}")

Collection name: my_recipes1
Collection metadata: {'description': 'My favorite recipes'}


In [7]:
# Adding Metadata
# Our actual data - the text we want to search through
recipes = [
    "Mix flour, eggs, and milk to make pancakes",
    "Boil pasta and add tomato sauce with basil",
    "Blend strawberries, banana, and yogurt for a smoothie",
    "Grill chicken with lemon and herbs",
    "Bake chocolate cake with vanilla frosting"
]

# Metadata - extra information about each document
# This is like adding columns to describe each row
recipe_metadata = [
    {
        "dish_name": "Pancakes",
        "meal_type": "breakfast",
        "prep_time_minutes": 20,
        "difficulty": "easy",
        "vegetarian": True
    },
    {
        "dish_name": "Pasta",
        "meal_type": "dinner",
        "prep_time_minutes": 30,
        "difficulty": "medium",
        "vegetarian": True
    },
    {
        "dish_name": "Smoothie",
        "meal_type": "breakfast",
        "prep_time_minutes": 5,
        "difficulty": "easy",
        "vegetarian": True
    },
    {
        "dish_name": "Grilled Chicken",
        "meal_type": "dinner",
        "prep_time_minutes": 45,
        "difficulty": "medium",
        "vegetarian": False
    },
    {
        "dish_name": "Chocolate Cake",
        "meal_type": "dessert",
        "prep_time_minutes": 60,
        "difficulty": "hard",
        "vegetarian": True
    }
]

# Unique IDs for each document - you choose these!
ids = ["recipe_1", "recipe_2", "recipe_3", "recipe_4", "recipe_5"]

# Add everything to the collection
collection.add(
    documents=recipes,      # The text
    metadatas=recipe_metadata,  # Extra information
    ids=ids                 # Unique identifiers
)

print(f"Successfully added {collection.count()} recipes!")

Successfully added 5 recipes!


In [8]:
# Get all documents from the collection
all_data = collection.get()

# Print everything
print("All documents in collection:")
print(all_data)
# for i in range(len(all_data['ids'])):
#     print(f"\nID: {all_data['ids'][i]}")
#     print(f"Document: {all_data['documents'][i]}")
#     print(f"Metadata: {all_data['metadatas'][i]}")

All documents in collection:
{'ids': ['recipe_1', 'recipe_2', 'recipe_3', 'recipe_4', 'recipe_5'], 'embeddings': None, 'documents': ['Mix flour, eggs, and milk to make pancakes', 'Boil pasta and add tomato sauce with basil', 'Blend strawberries, banana, and yogurt for a smoothie', 'Grill chicken with lemon and herbs', 'Bake chocolate cake with vanilla frosting'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'difficulty': 'easy', 'dish_name': 'Pancakes', 'meal_type': 'breakfast', 'vegetarian': True, 'prep_time_minutes': 20}, {'prep_time_minutes': 30, 'difficulty': 'medium', 'meal_type': 'dinner', 'dish_name': 'Pasta', 'vegetarian': True}, {'difficulty': 'easy', 'meal_type': 'breakfast', 'prep_time_minutes': 5, 'dish_name': 'Smoothie', 'vegetarian': True}, {'dish_name': 'Grilled Chicken', 'difficulty': 'medium', 'meal_type': 'dinner', 'vegetarian': False, 'prep_time_minutes': 45}, {'meal_type': 'dessert', 'vegetarian': True, 'difficulty': 'hard', 'pr

In [9]:
# Let's See How the Data is Stored
#ChromaDB uses SQLite internally to store metadata

#Vectors are stored in binary format for efficiency

#The folder structure is how ChromaDB organizes your collections

# Check what files were created
import os

db_path = "./my_chroma_db"
print("Files and folders created:")
for root, dirs, files in os.walk(db_path):
    level = root.replace(db_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')

Files and folders created:
my_chroma_db/
  chroma.sqlite3
  9bfbb7f8-009e-47c0-97f6-764c580d3f91/
    header.bin
    data_level0.bin
    length.bin
    link_lists.bin


In [9]:
# Fetching Data by ID
# Get specific recipes by their IDs
result = collection.get(
    ids=["recipe_1", "recipe_3"],  # Which IDs to fetch
    include=["documents", "metadatas"]  # What information to return
)

print("Fetched recipes:")
for i, doc in enumerate(result['documents']):
    print(f"\nRecipe {i+1}:")
    print(f"  Instructions: {doc}")
    print(f"  Metadata: {result['metadatas'][i]}")

Fetched recipes:

Recipe 1:
  Instructions: Mix flour, eggs, and milk to make pancakes
  Metadata: {'prep_time_minutes': 20, 'vegetarian': True, 'difficulty': 'easy', 'meal_type': 'breakfast', 'dish_name': 'Pancakes'}

Recipe 2:
  Instructions: Blend strawberries, banana, and yogurt for a smoothie
  Metadata: {'vegetarian': True, 'difficulty': 'easy', 'dish_name': 'Smoothie', 'meal_type': 'breakfast', 'prep_time_minutes': 5}


In [10]:
# Semantic Search (The Main Power)
# Search for recipes even with different words!
query = "I want something sweet after dinner"

print(f"Searching for: '{query}'")
print("=" * 50)

results = collection.query(
    query_texts=[query],  # What you're looking for
    n_results=2,  # How many results to return
    include=["documents", "metadatas", "distances"]
)

# Display results
for i in range(len(results['documents'][0])):
    print(f"\nMatch #{i+1}:")
    print(f"  Recipe: {results['documents'][0][i]}")
    print(f"  Dish: {results['metadatas'][0][i]['dish_name']}")
    print(f"  Meal type: {results['metadatas'][0][i]['meal_type']}")
    print(f"  Similarity distance: {results['distances'][0][i]:.4f}")


Searching for: 'I want something sweet after dinner'

Match #1:
  Recipe: Blend strawberries, banana, and yogurt for a smoothie
  Dish: Smoothie
  Meal type: breakfast
  Similarity distance: 0.6453

Match #2:
  Recipe: Grill chicken with lemon and herbs
  Dish: Grilled Chicken
  Meal type: dinner
  Similarity distance: 0.6470
